# Evaluación Sumativa 3 — Taller II | Grupo 2
**MCDI502 · Gestión de Datos y Tecnologías Big Data · Magíster en Ciencia de Datos e IA**

**Integrantes:** Verónica Durán · Raúl Moya · Daniela Rojas · Manuel Sánchez

**Objetivo.** Integrar cuatro fuentes heterogéneas (OpenMeteo, AQICN, MongoDB y MySQL) en Apache Spark, aplicar transformaciones de limpieza y clasificación, unificar los datos y responder siete consultas con Spark SQL. Entorno: VS Code en Windows, Python 3.13, Java 17 y PySpark 4.0.5.

> **Integridad de los datos:** no se duplican lecturas ni se imputan contaminantes ausentes con cero. AQICN publica el estado actual de cada estación (no un histórico), por lo que las 100 observaciones corresponden a 100 estaciones distintas.

> **Seguridad:** la URI de MongoDB y el token de AQICN no se guardan en el notebook ni en GitHub. Se leen desde las variables de entorno `MONGO_URI` y `AQICN_TOKEN`.

**Secciones:** entorno → ingesta → calidad y transformaciones → integración → consultas C1–C7 → validación final.

## 1. Configuración del entorno — VS Code (Windows)

**Equipo del Grupo 2:** Python 3.13, Java 17 y PySpark 4.0.5. Reutilizamos el entorno virtual de `Formativa 3 - APIs y Spark`, ya comprobado con Spark y MongoDB. **No usar Google Colab.**

**Seleccionar kernel:** en la esquina superior derecha de VS Code, `Select Kernel` → `Python Environments` → elegir el intérprete que termina en `Formativa 3 - APIs y Spark\.venv\Scripts\python.exe`. La carpeta de trabajo puede ser `Sumativa 3 - Taller II`; el entorno Python está en otra carpeta y eso es normal.

**Credenciales:** `MONGO_URI` y `AQICN_TOKEN` se pueden establecer en PowerShell antes de iniciar VS Code, o ingresar de forma oculta cuando el notebook las solicite. No escribirlas en celdas ni subirlas a GitHub.

**Importante:** la unidad virtual `S:` debe apuntar a la carpeta de la Formativa 3 (la que contiene `.venv`), **no** directamente a `.venv`. La celda 1.2 intenta prepararla automáticamente en Windows.


In [1]:
# 1.1 Verificación del intérprete y dependencias — compatible con VS Code
import os, sys, platform, subprocess, importlib.util
from datetime import datetime
from pathlib import Path

print('Python:', sys.version.split()[0])
print('Kernel:', sys.executable)
print('Sistema:', platform.system())

# Este es el entorno probado previamente para el proyecto.
RUTA_VENV = (Path.home() / 'Desktop' / 'ESTUDIOS' / 'MAGÍSTER' /
             '3 GESTIÓN DE DATOS Y TECNOLOGÍAS BIG DATA' /
             'Formativa 3 - APIs y Spark' / '.venv')
PYTHON_PROBADO = RUTA_VENV / 'Scripts' / 'python.exe'
if os.name == 'nt' and PYTHON_PROBADO.exists():
    if Path(sys.executable).resolve() != PYTHON_PROBADO.resolve():
        print('AVISO: VS Code no está usando el entorno probado de Formativa 3.')
        print('Selecciona este kernel:', PYTHON_PROBADO)

# Si se abrió otro kernel, instalar paquetes faltantes en ESE kernel, no en otro Python.
DEPENDENCIAS = {'requests':'requests', 'pandas':'pandas', 'pymongo':'pymongo',
                'dns':'dnspython', 'pyspark':'pyspark==4.0.5'}
faltantes = [paquete for modulo, paquete in DEPENDENCIAS.items()
             if importlib.util.find_spec(modulo) is None]
if faltantes:
    print('Faltan paquetes en este kernel:', ', '.join(faltantes))
    print('Instalando únicamente los paquetes faltantes...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *faltantes])
    print('Instalación completada. Si VS Code lo solicita, reinicia el kernel y ejecuta de nuevo.')

import requests
import pandas as pd
import pymongo
import pyspark
print('Requests:', requests.__version__, '| PyMongo:', pymongo.version,
      '| PySpark:', pyspark.__version__)

# Importaciones centralizadas para evitar NameError en ejecuciones parciales
import time
from pyspark.sql import functions as F
from pyspark.sql.functions import col, lit, when, round as spark_round


Python: 3.13.15
Kernel: C:\Users\raulm\AppData\Local\Programs\Python\Python313\python.exe
Sistema: Windows


Requests: 2.34.2 | PyMongo: 4.20.0 | PySpark: 4.0.5


In [2]:
# 1.1b Verificación de Java (requisito de Apache Spark)
import shutil
java = shutil.which('java')
java_home = os.environ.get('JAVA_HOME')
print('JAVA_HOME:', java_home or '(no definido; Spark usará el java del PATH)')
if java is None and java_home:
    candidato = Path(java_home) / 'bin' / ('java.exe' if os.name == 'nt' else 'java')
    java = str(candidato) if candidato.exists() else None
if java is None:
    raise RuntimeError('Java no está instalado. Instala JDK 17 (por ejemplo, Eclipse Temurin) '
                       'y define JAVA_HOME antes de crear la SparkSession.')
version_java = subprocess.run([java, '-version'], capture_output=True, text=True)
print('Java:', (version_java.stderr or version_java.stdout).splitlines()[0])

JAVA_HOME: C:/Program Files/Eclipse Adoptium/jdk-17.0.21.9-hotspot
Java: openjdk version "17.0.21" 2026-10-20


In [3]:
# 1.2 Configuración de Spark local: solución de rutas con espacios y tildes en Windows
import os, sys, subprocess
from pathlib import Path

if os.name == 'nt':
    # S: apunta a la carpeta «Formativa 3 - APIs y Spark» (contiene .venv).
    carpeta_formativa = RUTA_VENV.parent
    python_s = Path('S:/.venv/Scripts/python.exe')
    spark_s = Path('S:/.venv/Lib/site-packages/pyspark')
    if carpeta_formativa.is_dir() and not python_s.exists():
        resultado = subprocess.run(['subst', 'S:', str(carpeta_formativa)],
                                   capture_output=True, text=True, shell=True)
        if resultado.returncode != 0:
            print('Aviso: no fue posible crear S:.', resultado.stderr.strip())
    if python_s.exists() and spark_s.exists():
        os.environ['SPARK_HOME'] = str(spark_s)
        os.environ['PYSPARK_PYTHON'] = str(python_s)
        os.environ['PYSPARK_DRIVER_PYTHON'] = str(python_s)
        print('Spark configurado mediante unidad virtual S:')
    else:
        os.environ['PYSPARK_PYTHON'] = sys.executable
        print('Usando Python del kernel; si Spark falla, comprueba la unidad S:.')

from pyspark.sql import SparkSession, Window, functions as F
from pyspark.sql.types import (StructType, StructField, StringType, DoubleType,
                               IntegerType, TimestampType)
from pymongo import MongoClient

# Solicitud interactiva y oculta si la variable no llegó desde PowerShell al kernel.
# getpass evita imprimir los secretos en las salidas del notebook.
from getpass import getpass
for variable, descripcion in [('MONGO_URI','URI de MongoDB'),
                              ('AQICN_TOKEN','token personal de AQICN')]:
    if not os.environ.get(variable):
        valor = getpass(f'Ingresa {descripcion} (entrada oculta): ').strip()
        if valor:
            os.environ[variable] = valor
print('Conexiones configuradas:', {v:bool(os.getenv(v))
      for v in ['MONGO_URI','AQICN_TOKEN']})


Usando Python del kernel; si Spark falla, comprueba la unidad S:.
Conexiones configuradas: {'MONGO_URI': True, 'AQICN_TOKEN': True}


In [4]:
# 1.3 Inicialización de SparkSession / SparkContext
spark = (SparkSession.builder.master('local[2]')
         .appName('MCDI502_Sumativa3_Grupo2')
         .config('spark.sql.session.timeZone', 'America/Santiago')
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel('ERROR')
print('Spark:', spark.version, '| Aplicación:', sc.appName, '| Master:', sc.master)


Spark: 4.0.5 | Aplicación: MCDI502_Sumativa3_Grupo2 | Master: local[2]


## 2. Ingesta de datos de cuatro fuentes
Se conservan los nombres obligatorios `df_openmeteo`, `df_aqicn`, `df_mongo_bpm` y `df_mysql_dht`. Se seleccionan **hasta 100 observaciones auténticas** por fuente. Para OpenMeteo, MongoDB y MySQL, las pruebas previas confirmaron más de 100. AQICN requiere validación especial: cada respuesta `/feed/@id/` es una lectura de una estación, no un histórico de 100 momentos.
**Región AQICN:** priorizamos estaciones de Chile; como el país publica alrededor de 60 estaciones activas, completamos las 100 observaciones con estaciones de **Sudamérica y Latinoamérica**, la región más cercana. Solo si aun así faltaran, se recurre a otras regiones. La columna `estacion` permite auditar el origen de cada lectura.

**Nota AQICN:** los valores `iaqi` corresponden a subíndices de calidad del aire publicados por la API; no se interpretan automáticamente como concentraciones físicas en µg/m³.

In [5]:
# 2.1 OpenMeteo - Observaciones horarias reales de Concepción

from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType
)

LATITUD, LONGITUD = -36.827, -73.05

VARIABLES_OM = [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "wind_speed_10m",
    "wind_direction_10m",
    "uv_index"
]

# La API de archivo (archive-api) no publica uv_index: lo devolvía 100 % nulo.
# La API forecast con past_days entrega las 6 variables para los últimos 7 días;
# forecast_days=0 evita incluir horas futuras (pronóstico).
respuesta = requests.get(
    "https://api.open-meteo.com/v1/forecast",
    params={
        "latitude": LATITUD,
        "longitude": LONGITUD,
        "hourly": ",".join(VARIABLES_OM),
        "past_days": 7,
        "forecast_days": 0,
        "timezone": "America/Santiago"
    },
    timeout=30
)

respuesta.raise_for_status()
h = respuesta.json()["hourly"]

filas_om = []

for i, fecha in enumerate(h["time"]):
    fila = {
        "tiempo": fecha.replace("T", " ") + ":00"
    }

    for variable in VARIABLES_OM:
        valor = h[variable][i]

        # Convertir los valores numéricos a float.
        # Mantener None cuando falten datos.
        fila[variable] = (
            float(valor) if valor is not None else None
        )

    filas_om.append(fila)

# Seleccionar las 100 observaciones más recientes
filas_om = sorted(
    filas_om,
    key=lambda f: f["tiempo"],
    reverse=True
)[:100]

schema_om = StructType(
    [StructField("tiempo", StringType(), True)]
    + [
        StructField(v, DoubleType(), True)
        for v in VARIABLES_OM
    ]
)

df_openmeteo = spark.createDataFrame(
    filas_om,
    schema=schema_om
)

print("OpenMeteo:", df_openmeteo.count(), "registros horarios reales")
df_openmeteo.show(5, truncate=False)
# Auditoría: confirmamos que el índice UV llegue con datos (antes venía 100 % nulo).
uv_validos = df_openmeteo.filter(F.col('uv_index').isNotNull()).count()
print(f'UV disponible: {uv_validos}/100 registros.')
if uv_validos == 0:
    print('ADVERTENCIA: OpenMeteo no entregó índice UV en este período; se preservan NULL, no se inventan valores.')

OpenMeteo: 100 registros horarios reales


+-------------------+--------------+--------------------+-------------+--------------+------------------+--------+
|tiempo             |temperature_2m|relative_humidity_2m|precipitation|wind_speed_10m|wind_direction_10m|uv_index|
+-------------------+--------------+--------------------+-------------+--------------+------------------+--------+
|2026-11-14 23:00:00|16.3          |89.0                |0.0          |13.9          |14.0              |0.0     |
|2026-11-14 22:00:00|16.7          |86.0                |0.0          |16.2          |9.0               |0.0     |
|2026-11-14 21:00:00|17.2          |85.0                |0.0          |19.1          |3.0               |0.05    |
|2026-11-14 20:00:00|17.2          |84.0                |0.0          |18.2          |2.0               |0.75    |
|2026-11-14 19:00:00|18.1          |81.0                |0.0          |20.7          |1.0               |2.0     |
+-------------------+--------------+--------------------+-------------+---------

UV disponible: 100/100 registros.


In [6]:
# 2.2 AQICN — mediciones reales de estaciones diferentes; Chile primero, ampliación mundial si hace falta
# IMPORTANTE: la API /feed/ devuelve una instantánea por estación, no un histórico de 100 horas.
# Se identifican estaciones por UID y fecha, sin duplicar resultados ni inventar contaminantes.
import time
TOKEN_AQICN = os.getenv('AQICN_TOKEN')
if not TOKEN_AQICN:
    raise RuntimeError('Configura AQICN_TOKEN en la celda 1.2; no publiques tu token.')

CIUDADES_CHILE = ['Santiago','Valparaiso','Temuco','Concepcion','Talca','Rancagua',
    'Puerto Montt','Antofagasta','Chillan','Los Angeles','Arica','Iquique',
    'Calama','Copiapo','La Serena','Coquimbo','Quillota','San Antonio',
    'Curico','Linares','Constitucion','Coronel','Talcahuano','Hualpen',
    'Lota','Arauco','Angol','Victoria','Valdivia','Osorno','Coyhaique','Punta Arenas']
# Si Chile no alcanza 100 estaciones, se completa con la región más cercana
# (Sudamérica y Latinoamérica); el resto del mundo queda solo como último recurso.
CIUDADES_REGION = ['Buenos Aires','Mendoza','Cordoba Argentina','Rosario','Montevideo',
    'Lima','Arequipa','La Paz','Quito','Bogota','Medellin','Cali','Caracas',
    'Sao Paulo','Rio de Janeiro','Belo Horizonte','Curitiba','Porto Alegre',
    'Mexico City','Guadalajara','Monterrey']
CIUDADES_MUNDO = ['Madrid','Paris','London','New York','Toronto','Tokyo','Seoul','Sydney']

sesion_aqi = requests.Session()
estaciones = {}

def buscar_estaciones(ciudades, solo_chile=False):
    for ciudad in ciudades:
        if len(estaciones) >= 180: break
        try:
            r = sesion_aqi.get('https://api.waqi.info/search/',
                params={'keyword':ciudad,'token':TOKEN_AQICN}, timeout=15)
            r.raise_for_status()
            payload = r.json()
            if payload.get('status') != 'ok': continue
            for item in payload.get('data', []):
                nombre = item.get('station', {}).get('name', '')
                uid = item.get('uid')
                if uid is None or not nombre: continue
                if solo_chile and 'Chile' not in nombre: continue
                try: estaciones[int(uid)] = nombre
                except (ValueError,TypeError): continue
        except (requests.RequestException,ValueError,TypeError) as exc:
            print('Búsqueda no disponible:', ciudad, type(exc).__name__)

buscar_estaciones(CIUDADES_CHILE, solo_chile=True)
numero_chile = len(estaciones)
if numero_chile < 100:
    buscar_estaciones(CIUDADES_REGION)
if len(estaciones) < 100:
    buscar_estaciones(CIUDADES_MUNDO)

lecturas = {}
for uid,nombre in estaciones.items():
    if len(lecturas) >= 100: break
    try:
        r = sesion_aqi.get(f'https://api.waqi.info/feed/@{uid}/',
            params={'token':TOKEN_AQICN},timeout=15)
        r.raise_for_status()
        payload = r.json()
        if payload.get('status') != 'ok' or not isinstance(payload.get('data'),dict): continue
        info = payload['data']
        fecha = info.get('time',{}).get('s')
        if not fecha: continue
        iaqi = info.get('iaqi',{})
        def valor(clave):
            v = iaqi.get(clave,{}).get('v')
            return float(v) if isinstance(v,(int,float)) else None
        registro = {'estacion_id':uid,'estacion':nombre,'fecha_aqicn':fecha,
            'PM2_5':valor('pm25'),'PM10':valor('pm10'),'O3':valor('o3'),
            'NO2':valor('no2'),'SO2':valor('so2')}
        if any(registro[c] is not None for c in ('PM2_5','PM10','O3','NO2','SO2')):
            lecturas[(uid,fecha)] = registro
    except (requests.RequestException,ValueError,TypeError) as exc:
        print('Estación no disponible:',uid,type(exc).__name__)
    time.sleep(0.08)

filas_aqi = list(lecturas.values())[:100]
schema_aqi = StructType([
    StructField('estacion_id',IntegerType()),StructField('estacion',StringType()),
    StructField('fecha_aqicn',StringType()),
    *[StructField(c,DoubleType()) for c in ['PM2_5','PM10','O3','NO2','SO2']]
])
df_aqicn = spark.createDataFrame(filas_aqi,schema_aqi)
print('Estaciones chilenas identificadas:',numero_chile)
print('Estaciones candidatas totales:',len(estaciones))
print('Observaciones reales distintas:',df_aqicn.count())
print('Cobertura geográfica: Chile primero, luego Sudamérica y Latinoamérica.')
if df_aqicn.count() < 100:
    print('BRECHA AQICN:',100-df_aqicn.count(),
          'observaciones; no se duplican ni fabrican datos.')
df_aqicn.show(10,truncate=False)

Estaciones chilenas identificadas: 62
Estaciones candidatas totales: 186


Observaciones reales distintas: 100
Cobertura geográfica: Chile primero, luego Sudamérica y Latinoamérica.


+-----------+-------------------------+-------------------+-----+-----+----+----+----+
|estacion_id|estacion                 |fecha_aqicn        |PM2_5|PM10 |O3  |NO2 |SO2 |
+-----------+-------------------------+-------------------+-----+-----+----+----+----+
|435        |Cerro Navia, Chile       |2026-11-09 21:00:00|50.0 |26.0 |8.8 |11.7|NULL|
|421        |La Florida, Chile        |2026-11-09 20:00:00|42.0 |36.0 |18.4|13.6|3.0 |
|14731      |Cerrillos II, Chile      |2026-11-09 20:00:00|25.0 |38.0 |NULL|NULL|NULL|
|431        |Parque O'Higgins, Chile  |2026-11-09 20:00:00|25.0 |29.0 |18.4|12.7|NULL|
|433        |Cerrillos, Chile         |2022-04-26 16:00:00|34.0 |104.0|11.2|14.3|38.0|
|429        |Independencia, Chile     |2021-08-02 03:00:00|13.0 |1.0  |24.8|36.9|38.0|
|4974       |Valparaiso, Chile        |2026-11-09 21:00:00|55.0 |NULL |NULL|NULL|NULL|
|4978       |Viña del Mar, Chile      |2026-11-09 20:00:00|17.0 |22.0 |NULL|NULL|NULL|
|425        |Las Encinas Temuco, Chile|2026

In [7]:
# 2.3 MongoDB — 100 documentos auténticos; PyMongo evita conectores incompatibles
MONGO_URI = os.getenv('MONGO_URI')
if not MONGO_URI:
    raise RuntimeError('Falta MONGO_URI. Configúrala en PowerShell y reinicia el kernel.')
cliente = MongoClient(MONGO_URI, serverSelectionTimeoutMS=12000)
try:
    cliente.admin.command('ping')
    coleccion = cliente['telemetria']['lecturas']
    cursor = coleccion.find({
        'altitudFt':{'$ne':None}, 'presionAtm':{'$ne':None}, 'ppmCO':{'$ne':None}
    }, {'_id':0,'fecha':1,'altitudFt':1,'presionAtm':1,'presionPa':1,
        'ppmCO':1,'temperatura':1}).sort('fecha',-1).limit(100)
    filas_mongo = []
    for x in cursor:
        filas_mongo.append({
            'fecha':x.get('fecha'),
            'alturaft':float(x['altitudFt']),
            'presionAtm':float(x['presionAtm']),
            'presionPa':float(x['presionPa']) if x.get('presionPa') is not None else None,
            'ppmCO':float(x['ppmCO']),
            'temperatura_bmp':float(x['temperatura']) if x.get('temperatura') is not None else None
        })
finally:
    cliente.close()
schema_mongo = StructType([
    StructField('fecha',TimestampType()),StructField('alturaft',DoubleType()),
    StructField('presionAtm',DoubleType()),StructField('presionPa',DoubleType()),
    StructField('ppmCO',DoubleType()),StructField('temperatura_bmp',DoubleType())
])
df_mongo_bpm = spark.createDataFrame(filas_mongo,schema_mongo)
print('MongoDB:',df_mongo_bpm.count(),'registros')
df_mongo_bpm.show(5,truncate=False)


MongoDB: 100 registros


+-----------------------+--------+----------+---------+-----+---------------+
|fecha                  |alturaft|presionAtm|presionPa|ppmCO|temperatura_bmp|
+-----------------------+--------+----------+---------+-----+---------------+
|2026-11-15 08:37:23.037|837.58  |0.970126  |98298.0  |3.76 |23.4           |
|2026-11-15 08:37:06.053|836.74  |0.970106  |98296.0  |3.73 |23.5           |
|2026-11-15 08:36:49.188|839.26  |0.970027  |98288.0  |3.73 |23.5           |
|2026-11-15 08:36:32.372|838.42  |0.970116  |98297.0  |3.65 |23.4           |
|2026-11-15 08:36:15.464|835.06  |0.970136  |98299.0  |3.76 |23.4           |
+-----------------------+--------+----------+---------+-----+---------------+
only showing top 5 rows


In [8]:
# 2.4 MySQL vía API PHP — 100 lecturas reales y diferentes
URL_MYSQL_PHP = 'http://mgtt.comunidadingenieria.cl/obtener_datos.php'
r = requests.get(URL_MYSQL_PHP,timeout=30)
r.raise_for_status()
raw_mysql = r.json()
if not isinstance(raw_mysql,list):
    raise ValueError('La API MySQL no devolvió una lista JSON')
filas_mysql = []
ids = set()
for x in sorted(raw_mysql,key=lambda a:int(a['id']),reverse=True):
    if int(x['id']) in ids: continue
    ids.add(int(x['id']))
    filas_mysql.append({'id':int(x['id']),'fecha':x.get('fecha'),
                        'temperatura_interior':float(x['temperatura']) if x.get('temperatura') is not None else None,
                        'humedad':float(x['humedad']) if x.get('humedad') is not None else None})
    if len(filas_mysql) == 100: break
schema_mysql = StructType([
    StructField('id',IntegerType()),StructField('fecha',StringType()),
    StructField('temperatura_interior',DoubleType()),StructField('humedad',DoubleType())
])
df_mysql_dht = spark.createDataFrame(filas_mysql,schema_mysql)
print('MySQL:',df_mysql_dht.count(),'registros')
df_mysql_dht.show(5,truncate=False)


MySQL: 100 registros


+------+-------------------+--------------------+-------+
|id    |fecha              |temperatura_interior|humedad|
+------+-------------------+--------------------+-------+
|379254|2026-11-15 00:37:22|21.0                |68.4   |
|379253|2026-11-15 00:37:05|21.1                |68.2   |
|379252|2026-11-15 00:36:49|21.1                |68.1   |
|379251|2026-11-15 00:36:32|21.1                |68.0   |
|379250|2026-11-15 00:36:15|21.1                |68.0   |
+------+-------------------+--------------------+-------+
only showing top 5 rows


---
# 3. Preprocesamiento y Transformación de Datos (RA4, ID4.2)

Siguiendo la metodología del apunte del curso (*Navarro, E. (2026). Preprocesamiento de datos: técnicas de limpieza y transformación*), estructuramos esta fase en tres etapas:
1. **Exploración y diagnóstico de calidad:** conteo de registros, nulos por columna y estadísticas descriptivas (`describe`) de cada fuente. **Decisión:** los nulos se conservan como dato faltante y no se imputan (`fillna`), porque reemplazar un contaminante ausente por 0 bajaría artificialmente los promedios; `AVG` de Spark SQL ignora los `NULL`.
2. **3.1 Columna calculada `altura_metros` en `df_mongo_bpm`:** conversión de pies a metros con el factor exacto $1\text{ pie} = 0.3048\text{ metros}$ sobre la columna `alturaft`.
3. **3.2 Columna de clasificación `condicion_clima` en `df_openmeteo`:** lógica condicional con `when().otherwise()`:
   - $\text{Temperatura 2m} > 30^\circ\text{C} \land \text{Humedad relativa 2m} > 70\% \implies \text{CALOR Y HUMEDAD}$
   - $\text{Temperatura 2m} > 30^\circ\text{C} \land \text{Humedad relativa 2m} \le 70\% \implies \text{CALOR SECO}$
   - $\text{Temperatura 2m} \le 30^\circ\text{C} \implies \text{CLIMA MODERADO}$
   - $\text{De lo contrario} \implies \text{CONDICION DESCONOCIDA}$

In [9]:
# 3.0 Auditoría de calidad (no reemplazar automáticamente nulos por cero)
def auditar_calidad(df,nombre):
    print('Calidad:',nombre,'| filas:',df.count())
    df.select([F.sum(F.col(c).isNull().cast('int')).alias(c) for c in df.columns]).show(truncate=False)
    numericas = [c for c, t in df.dtypes if t in ('double', 'int')]
    df.describe(*numericas).show(truncate=False)

for nombre, df in [('OpenMeteo',df_openmeteo),('AQICN',df_aqicn),
                   ('MongoDB',df_mongo_bpm),('MySQL',df_mysql_dht)]:
    auditar_calidad(df,nombre)
print('Los valores nulos se conservan como datos faltantes; AVG ignora NULL en Spark SQL.')

Calidad: OpenMeteo | filas: 100


+------+--------------+--------------------+-------------+--------------+------------------+--------+
|tiempo|temperature_2m|relative_humidity_2m|precipitation|wind_speed_10m|wind_direction_10m|uv_index|
+------+--------------+--------------------+-------------+--------------+------------------+--------+
|0     |0             |0                   |0            |0             |0                 |0       |
+------+--------------+--------------------+-------------+--------------+------------------+--------+



+-------+------------------+--------------------+-------------+------------------+------------------+-----------------+
|summary|temperature_2m    |relative_humidity_2m|precipitation|wind_speed_10m    |wind_direction_10m|uv_index         |
+-------+------------------+--------------------+-------------+------------------+------------------+-----------------+
|count  |100               |100                 |100          |100               |100               |100              |
|mean   |17.952999999999996|74.4                |0.0          |12.121000000000002|170.66            |2.4855           |
|stddev |3.3943641644361158|13.302995789283592  |0.0          |5.310011413558469 |83.03561606535796 |3.113892745471981|
|min    |12.8              |46.0                |0.0          |1.1               |1.0               |0.0              |
|max    |26.3              |96.0                |0.0          |24.8              |358.0             |8.7              |
+-------+------------------+------------

Calidad: AQICN | filas: 100


+-----------+--------+-----------+-----+----+---+---+---+
|estacion_id|estacion|fecha_aqicn|PM2_5|PM10|O3 |NO2|SO2|
+-----------+--------+-----------+-----+----+---+---+---+
|0          |0       |0          |9    |18  |48 |47 |49 |
+-----------+--------+-----------+-----+----+---+---+---+



+-------+-----------------+------------------+------------------+-----------------+-----------------+------------------+
|summary|estacion_id      |PM2_5             |PM10              |O3               |NO2              |SO2               |
+-------+-----------------+------------------+------------------+-----------------+-----------------+------------------+
|count  |100              |91                |82                |52               |53               |51                |
|mean   |7215.15          |39.30769230769231 |29.097560975609756|22.725           |9.915094339622641|7.070588235294116 |
|stddev |5035.948843569651|29.207568390436943|23.11143683322688 |18.43798163212738|7.990629036238325|12.343650904293218|
|min    |379              |1.0               |1.0               |0.7              |1.4              |0.2               |
|max    |14762            |193.0             |155.0             |94.4             |36.9             |67.0              |
+-------+-----------------+-----

... (2 outputs left out)

+-------+------------------+--------------------+----------------+-------------------+-------------------+
|summary|alturaft          |presionAtm          |presionPa       |ppmCO              |temperatura_bmp    |
+-------+------------------+--------------------+----------------+-------------------+-------------------+
|count  |100               |100                 |100             |100                |100                |
|mean   |833.8952          |0.9702351200000001  |98309.08        |3.731799999999999  |23.511000000000013 |
|stddev |2.3266104360661917|8.551629060945843E-5|8.66978964711486|0.03723743093483038|0.08396127053287553|
|min    |828.06            |0.970027            |98288.0         |3.62               |23.4               |
|max    |839.26            |0.970402            |98326.0         |3.84               |23.7               |
+-------+------------------+--------------------+----------------+-------------------+-------------------+



Calidad: MySQL | filas: 100


+---+-----+--------------------+-------+
|id |fecha|temperatura_interior|humedad|
+---+-----+--------------------+-------+
|0  |0    |0                   |0      |
+---+-----+--------------------+-------+



+-------+------------------+--------------------+-------------------+
|summary|id                |temperatura_interior|humedad            |
+-------+------------------+--------------------+-------------------+
|count  |100               |100                 |100                |
|mean   |379204.5          |21.154000000000007  |67.90199999999999  |
|stddev |29.011491975882016|0.10092501466860648 |0.31939968944278846|
|min    |379155            |21.0                |67.1               |
|max    |379254            |21.4                |68.4               |
+-------+------------------+--------------------+-------------------+

Los valores nulos se conservan como datos faltantes; AVG ignora NULL en Spark SQL.


In [10]:
# 3.1 Creación de la columna calculada altura_metros
# Conversión: 1 pie = 0.3048 metros

from pyspark.sql.functions import col, lit, round as spark_round

FACTOR_PIE_A_METRO = 0.3048

df_mongo_bpm = df_mongo_bpm.withColumn(
    "altura_metros",
    spark_round(
        col("alturaft") * lit(FACTOR_PIE_A_METRO),
        4
    )
)

print("Columna altura_metros creada correctamente.")

df_mongo_bpm.select(
    "fecha",
    "alturaft",
    "altura_metros",
    "presionAtm",
    "ppmCO"
).show(10, truncate=False)

Columna altura_metros creada correctamente.


+-----------------------+--------+-------------+----------+-----+
|fecha                  |alturaft|altura_metros|presionAtm|ppmCO|
+-----------------------+--------+-------------+----------+-----+
|2026-11-15 08:37:23.037|837.58  |255.2944     |0.970126  |3.76 |
|2026-11-15 08:37:06.053|836.74  |255.0384     |0.970106  |3.73 |
|2026-11-15 08:36:49.188|839.26  |255.8064     |0.970027  |3.73 |
|2026-11-15 08:36:32.372|838.42  |255.5504     |0.970116  |3.65 |
|2026-11-15 08:36:15.464|835.06  |254.5263     |0.970136  |3.76 |
|2026-11-15 08:35:58.462|834.5   |254.3556     |0.970244  |3.73 |
|2026-11-15 08:35:41.637|835.34  |254.6116     |0.970155  |3.76 |
|2026-11-15 08:35:24.814|835.06  |254.5263     |0.970195  |3.73 |
|2026-11-15 08:35:07.959|837.86  |255.3797     |0.970155  |3.76 |
|2026-11-15 08:34:51.013|836.74  |255.0384     |0.970136  |3.73 |
+-----------------------+--------+-------------+----------+-----+
only showing top 10 rows


In [11]:
# 3.2 Clasificación de la condición climática

from pyspark.sql.functions import col, lit, when

df_openmeteo = df_openmeteo.withColumn(
    "condicion_clima",
    when(
        (col("temperature_2m") > 30.0) &
        (col("relative_humidity_2m") > 70.0),
        lit("CALOR Y HUMEDAD")
    )
    .when(
        (col("temperature_2m") > 30.0) &
        (col("relative_humidity_2m") <= 70.0),
        lit("CALOR SECO")
    )
    .when(
        col("temperature_2m") <= 30.0,
        lit("CLIMA MODERADO")
    )
    .otherwise(lit("CONDICION DESCONOCIDA"))
)

print("Columna condicion_clima creada correctamente.")

df_openmeteo.select(
    "tiempo",
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "condicion_clima"
).show(10, truncate=False)

Columna condicion_clima creada correctamente.


+-------------------+--------------+--------------------+-------------+---------------+
|tiempo             |temperature_2m|relative_humidity_2m|precipitation|condicion_clima|
+-------------------+--------------+--------------------+-------------+---------------+
|2026-11-14 23:00:00|16.3          |89.0                |0.0          |CLIMA MODERADO |
|2026-11-14 22:00:00|16.7          |86.0                |0.0          |CLIMA MODERADO |
|2026-11-14 21:00:00|17.2          |85.0                |0.0          |CLIMA MODERADO |
|2026-11-14 20:00:00|17.2          |84.0                |0.0          |CLIMA MODERADO |
|2026-11-14 19:00:00|18.1          |81.0                |0.0          |CLIMA MODERADO |
|2026-11-14 18:00:00|18.8          |77.0                |0.0          |CLIMA MODERADO |
|2026-11-14 17:00:00|19.1          |75.0                |0.0          |CLIMA MODERADO |
|2026-11-14 16:00:00|19.7          |73.0                |0.0          |CLIMA MODERADO |
|2026-11-14 15:00:00|20.1       

---
## 4. Integración de datos (`df_unificado`)
**Decisión metodológica:** las fuentes no comparten una clave de evento ni tienen la misma frecuencia de muestreo. Por tanto, un `JOIN` exacto por fecha no es válido. Para ofrecer una **vista comparativa** se construye una unión horizontal por posición, ordenando cada fuente por una clave estable y asignando `id_secuencia`.

**Limitación fundamental:** compartir un número de fila **no significa** que los eventos sean simultáneos, de la misma ubicación ni causalmente relacionados. La tabla resultante sirve para mostrar variables juntas y practicar joins; **no debe utilizarse para calcular correlaciones físicas entre sensores**. Se conservan las fechas y estaciones de origen para auditoría. Se usa `full_outer` para no descartar datos si AQICN aporta menos de 100 observaciones.


In [12]:
# 4.1 Join posicional descriptivo; orden estable y trazabilidad
om = (df_openmeteo.withColumn('id_secuencia',F.row_number().over(Window.orderBy('tiempo')))
      .withColumnRenamed('tiempo','fecha_openmeteo'))
aqi = (df_aqicn.withColumn('id_secuencia',F.row_number().over(
        Window.orderBy('fecha_aqicn','estacion_id'))))
mongo = (df_mongo_bpm.withColumn('id_secuencia',F.row_number().over(Window.orderBy('fecha')))
         .withColumnRenamed('fecha','fecha_mongo'))
mysql = (df_mysql_dht.withColumn('id_secuencia',F.row_number().over(Window.orderBy('id')))
         .withColumnRenamed('fecha','fecha_mysql')
         .withColumnRenamed('humedad','humedad_interior'))
df_unificado = (om.join(aqi,'id_secuencia','full_outer')
               .join(mongo,'id_secuencia','full_outer')
               .join(mysql,'id_secuencia','full_outer')
               .orderBy('id_secuencia'))
print('df_unificado:',df_unificado.count(),'filas; columnas:',len(df_unificado.columns))
df_unificado.show(5,truncate=False)


df_unificado: 100 filas; columnas: 28


+------------+-------------------+--------------+--------------------+-------------+--------------+------------------+--------+---------------+-----------+-----------------------------------+-------------------+-----+----+----+----+----+-----------------------+--------+----------+---------+-----+---------------+-------------+------+-------------------+--------------------+----------------+
|id_secuencia|fecha_openmeteo    |temperature_2m|relative_humidity_2m|precipitation|wind_speed_10m|wind_direction_10m|uv_index|condicion_clima|estacion_id|estacion                           |fecha_aqicn        |PM2_5|PM10|O3  |NO2 |SO2 |fecha_mongo            |alturaft|presionAtm|presionPa|ppmCO|temperatura_bmp|altura_metros|id    |fecha_mysql        |temperatura_interior|humedad_interior|
+------------+-------------------+--------------+--------------------+-------------+--------------+------------------+--------+---------------+-----------+-----------------------------------+-------------------+---

---
# 5. Consultas y Extracción de Datos (ID3.1, ID4.1)

Tal como exigen las instrucciones y la rúbrica de evaluación, **primero registramos una tabla/vista temporal por cada uno de los cuatro DataFrames** utilizando `createOrReplaceTempView` en Spark SQL, y luego ejecutamos las **7 consultas obligatorias (`C1` a `C7`)**.

In [13]:
# 5.0 Creación y validación de tablas temporales en Spark SQL

tablas = {
    "df_openmeteo": df_openmeteo,
    "df_aqicn": df_aqicn,
    "df_mongo_bpm": df_mongo_bpm,
    "df_mysql_dht": df_mysql_dht,
    "df_unificado": df_unificado
}

for nombre, dataframe in tablas.items():
    dataframe.createOrReplaceTempView(nombre)
    print(f"Tabla registrada: {nombre}")

print("\nValidación de tablas temporales:")

for nombre in tablas:
    existe = spark.catalog.tableExists(nombre)
    print(f"{nombre}: {'OK' if existe else 'NO ENCONTRADA'}")

Tabla registrada: df_openmeteo
Tabla registrada: df_aqicn
Tabla registrada: df_mongo_bpm
Tabla registrada: df_mysql_dht
Tabla registrada: df_unificado

Validación de tablas temporales:
df_openmeteo: OK
df_aqicn: OK
df_mongo_bpm: OK
df_mysql_dht: OK
df_unificado: OK


In [14]:
# ==============================================================================
# Consulta C1 (df_openmeteo - ID4.1):
# "Mostrar los 10 registros ordenados por temperatura (de más alta a más baja)."
# ==============================================================================

consulta_c1 = spark.sql("""
    SELECT
        tiempo,
        temperature_2m,
        relative_humidity_2m,
        precipitation,
        wind_speed_10m,
        wind_direction_10m,
        uv_index,
        condicion_clima
    FROM df_openmeteo
    ORDER BY temperature_2m DESC
    LIMIT 10
""")

print("📌 RESULTADO CONSULTA C1: Top 10 registros con mayor temperatura (temperature_2m DESC)")
consulta_c1.show(10, truncate=False)

📌 RESULTADO CONSULTA C1: Top 10 registros con mayor temperatura (temperature_2m DESC)


+-------------------+--------------+--------------------+-------------+--------------+------------------+--------+---------------+
|tiempo             |temperature_2m|relative_humidity_2m|precipitation|wind_speed_10m|wind_direction_10m|uv_index|condicion_clima|
+-------------------+--------------+--------------------+-------------+--------------+------------------+--------+---------------+
|2026-11-13 15:00:00|26.3          |47.0                |0.0          |11.1          |239.0             |7.75    |CLIMA MODERADO |
|2026-11-13 16:00:00|26.0          |54.0                |0.0          |12.2          |232.0             |7.3     |CLIMA MODERADO |
|2026-11-13 17:00:00|25.5          |59.0                |0.0          |13.4          |228.0             |5.95    |CLIMA MODERADO |
|2026-11-13 14:00:00|25.4          |46.0                |0.0          |9.9           |243.0             |7.95    |CLIMA MODERADO |
|2026-11-13 13:00:00|24.8          |47.0                |0.0          |8.5         

In [15]:
# ==============================================================================
# Consulta C2 (df_openmeteo - ID4.1):
# "¿Cuál es el promedio de la humedad relativa (Humedad relativa 2m) para un mes
#  y un día en específico (elija el día y mes que ustedes deseen)?"
# ==============================================================================

fecha_muestra = spark.sql("""
    SELECT
        MONTH(TO_TIMESTAMP(tiempo)) AS mes_elegido,
        DAY(TO_TIMESTAMP(tiempo)) AS dia_elegido
    FROM df_openmeteo
    WHERE tiempo IS NOT NULL
    ORDER BY tiempo DESC
    LIMIT 1
""").collect()[0]

mes_sel = fecha_muestra["mes_elegido"]
dia_sel = fecha_muestra["dia_elegido"]

consulta_c2 = spark.sql(f"""
    SELECT
        MONTH(TO_TIMESTAMP(tiempo)) AS mes,
        DAY(TO_TIMESTAMP(tiempo)) AS dia,
        COUNT(*) AS total_registros_dia,
        ROUND(AVG(relative_humidity_2m), 2) AS promedio_humedad_relativa_2m
    FROM df_openmeteo
    WHERE MONTH(TO_TIMESTAMP(tiempo)) = {mes_sel}
      AND DAY(TO_TIMESTAMP(tiempo)) = {dia_sel}
    GROUP BY MONTH(TO_TIMESTAMP(tiempo)), DAY(TO_TIMESTAMP(tiempo))
""")

print(f"📌 RESULTADO CONSULTA C2: Promedio de Humedad Relativa 2m para el Día {dia_sel} del Mes {mes_sel}")
consulta_c2.show(truncate=False)

📌 RESULTADO CONSULTA C2: Promedio de Humedad Relativa 2m para el Día 14 del Mes 11


+---+---+-------------------+----------------------------+
|mes|dia|total_registros_dia|promedio_humedad_relativa_2m|
+---+---+-------------------+----------------------------+
|11 |14 |24                 |78.79                       |
+---+---+-------------------+----------------------------+



In [16]:
# C3: registros con precipitación > 2 mm en los últimos siete días de observaciones
# La fecha de referencia es la más reciente disponible en el DataFrame, no la fecha
# de ejecución del notebook (evita que una reejecución futura dé un falso cero).
consulta_c3 = spark.sql("""
    SELECT COUNT(*) AS registros_precipitacion_mayor_2mm
    FROM df_openmeteo
    WHERE precipitation > 2.0
      AND TO_TIMESTAMP(tiempo) >= (
          SELECT MAX(TO_TIMESTAMP(tiempo)) - INTERVAL 7 DAYS FROM df_openmeteo
      )
""")
print('C3: precipitación > 2 mm en los siete días anteriores al último registro')
consulta_c3.show(truncate=False)


C3: precipitación > 2 mm en los siete días anteriores al último registro


+---------------------------------+
|registros_precipitacion_mayor_2mm|
+---------------------------------+
|0                                |
+---------------------------------+



In [17]:
# ==============================================================================
# Consulta C4 (df_aqicn - ID4.1):
# "¿Cuál es el promedio de PM2.5, PM10, O3, NO2, SO2?"
# Nota: Se consulta directamente sobre PM2_5 (reemplazado el '.' por '_' según
# indicación del docente).
# ==============================================================================

consulta_c4 = spark.sql("""
    SELECT
        ROUND(AVG(PM2_5), 2) AS promedio_PM2_5,
        ROUND(AVG(PM10), 2)  AS promedio_PM10,
        ROUND(AVG(O3), 2)    AS promedio_O3,
        ROUND(AVG(NO2), 2)   AS promedio_NO2,
        ROUND(AVG(SO2), 2)   AS promedio_SO2
    FROM df_aqicn
""")

print("📌 RESULTADO CONSULTA C4: Promedio de contaminantes atmosféricos (PM2_5, PM10, O3, NO2, SO2)")
consulta_c4.show(truncate=False)

📌 RESULTADO CONSULTA C4: Promedio de contaminantes atmosféricos (PM2_5, PM10, O3, NO2, SO2)


+--------------+-------------+-----------+------------+------------+
|promedio_PM2_5|promedio_PM10|promedio_O3|promedio_NO2|promedio_SO2|
+--------------+-------------+-----------+------------+------------+
|39.31         |29.1         |22.73      |9.92        |7.07        |
+--------------+-------------+-----------+------------+------------+



In [18]:
# ==============================================================================
# Consulta C5 (df_aqicn - ID4.1):
# "¿Cuántos registros tuvieron niveles de NO2 superiores a 4?"
# ==============================================================================

consulta_c5 = spark.sql("""
    SELECT
        COUNT(*) AS registros_NO2_mayor_a_4
    FROM df_aqicn
    WHERE NO2 > 4.0
""")

print("📌 RESULTADO CONSULTA C5: Cantidad de registros en df_aqicn con niveles de NO2 > 4")
consulta_c5.show(truncate=False)

📌 RESULTADO CONSULTA C5: Cantidad de registros en df_aqicn con niveles de NO2 > 4


+-----------------------+
|registros_NO2_mayor_a_4|
+-----------------------+
|39                     |
+-----------------------+



In [19]:
# ==============================================================================
# Consulta C6 (df_mongo_bpm - ID3.1):
# "Muestre los primeros 10 registros del DataFrame, incluyendo la columna
#  calculada altura_metros"
# ==============================================================================

consulta_c6 = spark.sql("""
    SELECT
        fecha,
        alturaft,
        altura_metros,
        presionAtm,
        presionPa,
        ppmCO,
        temperatura_bmp
    FROM df_mongo_bpm
    ORDER BY fecha DESC
    LIMIT 10
""")

print("📌 RESULTADO CONSULTA C6: Primeros 10 registros de df_mongo_bpm incluyendo 'altura_metros'")
consulta_c6.show(10, truncate=False)

📌 RESULTADO CONSULTA C6: Primeros 10 registros de df_mongo_bpm incluyendo 'altura_metros'


+-----------------------+--------+-------------+----------+---------+-----+---------------+
|fecha                  |alturaft|altura_metros|presionAtm|presionPa|ppmCO|temperatura_bmp|
+-----------------------+--------+-------------+----------+---------+-----+---------------+
|2026-11-15 08:37:23.037|837.58  |255.2944     |0.970126  |98298.0  |3.76 |23.4           |
|2026-11-15 08:37:06.053|836.74  |255.0384     |0.970106  |98296.0  |3.73 |23.5           |
|2026-11-15 08:36:49.188|839.26  |255.8064     |0.970027  |98288.0  |3.73 |23.5           |
|2026-11-15 08:36:32.372|838.42  |255.5504     |0.970116  |98297.0  |3.65 |23.4           |
|2026-11-15 08:36:15.464|835.06  |254.5263     |0.970136  |98299.0  |3.76 |23.4           |
|2026-11-15 08:35:58.462|834.5   |254.3556     |0.970244  |98310.0  |3.73 |23.4           |
|2026-11-15 08:35:41.637|835.34  |254.6116     |0.970155  |98301.0  |3.76 |23.4           |
|2026-11-15 08:35:24.814|835.06  |254.5263     |0.970195  |98305.0  |3.73 |23.5 

In [20]:
# ==============================================================================
# Consulta C7 (df_openmeteo - ID3.1):
# "Muestre los primeros 10 registros del DataFrame, incluyendo la columna
#  calculada condicion_clima."
# ==============================================================================

consulta_c7 = spark.sql("""
    SELECT
        tiempo,
        temperature_2m,
        relative_humidity_2m,
        precipitation,
        wind_speed_10m,
        wind_direction_10m,
        uv_index,
        condicion_clima
    FROM df_openmeteo
    ORDER BY tiempo DESC
    LIMIT 10
""")

print("📌 RESULTADO CONSULTA C7: Primeros 10 registros de df_openmeteo incluyendo 'condicion_clima'")
consulta_c7.show(10, truncate=False)

📌 RESULTADO CONSULTA C7: Primeros 10 registros de df_openmeteo incluyendo 'condicion_clima'


+-------------------+--------------+--------------------+-------------+--------------+------------------+--------+---------------+
|tiempo             |temperature_2m|relative_humidity_2m|precipitation|wind_speed_10m|wind_direction_10m|uv_index|condicion_clima|
+-------------------+--------------+--------------------+-------------+--------------+------------------+--------+---------------+
|2026-11-14 23:00:00|16.3          |89.0                |0.0          |13.9          |14.0              |0.0     |CLIMA MODERADO |
|2026-11-14 22:00:00|16.7          |86.0                |0.0          |16.2          |9.0               |0.0     |CLIMA MODERADO |
|2026-11-14 21:00:00|17.2          |85.0                |0.0          |19.1          |3.0               |0.05    |CLIMA MODERADO |
|2026-11-14 20:00:00|17.2          |84.0                |0.0          |18.2          |2.0               |0.75    |CLIMA MODERADO |
|2026-11-14 19:00:00|18.1          |81.0                |0.0          |20.7        

---
## 6. Validación final y límites del análisis
- **Cuatro fuentes con 100 registros cada una**, verificado en la celda siguiente.
- **OpenMeteo:** usamos la API *forecast* con `past_days=7`, porque la API de archivo no entrega el índice UV. Así se capturan las seis variables solicitadas.
- **AQICN:** cada observación es la última medición publicada por una estación distinta; priorizamos Chile y completamos con Sudamérica y Latinoamérica. Algunas fechas pueden ser anteriores a la consulta. Los contaminantes ausentes permanecen como `NULL`; no se generan históricos artificiales.
- **Integración:** `df_unificado` es una tabla descriptiva por posición. Las fuentes no miden el mismo lugar ni el mismo instante, por lo que no se usa para inferir correlaciones físicas entre sensores.
- **Seguridad:** no se almacenan contraseñas ni tokens en el notebook.

In [21]:
# 6.1 Conteos y verificación final
resumen = [('df_openmeteo',df_openmeteo),('df_aqicn',df_aqicn),
           ('df_mongo_bpm',df_mongo_bpm),('df_mysql_dht',df_mysql_dht),
           ('df_unificado',df_unificado)]
for nombre,df in resumen:
    n_filas=df.count()
    print(f'{nombre}: {n_filas} filas, {len(df.columns)} columnas')
    if nombre != 'df_unificado' and n_filas < 100:
        print(f'  PENDIENTE: faltan {100-n_filas} observaciones reales para cumplir la meta.')
assert 'altura_metros' in df_mongo_bpm.columns
assert 'condicion_clima' in df_openmeteo.columns
assert df_openmeteo.count()==100, 'OpenMeteo debe tener 100 registros'
assert df_mongo_bpm.count()==100, 'MongoDB debe tener 100 registros'
assert df_mysql_dht.count()==100, 'MySQL debe tener 100 registros'
print('Transformaciones y tres fuentes con 100 registros: verificadas.')
if df_aqicn.count()!=100:
    print('ATENCIÓN: AQICN no alcanza 100 observaciones únicas; resolver antes de afirmar cumplimiento total.')

uv_no_nulos = df_openmeteo.filter(F.col('uv_index').isNotNull()).count()
print('OpenMeteo UV con datos:',uv_no_nulos,'de',df_openmeteo.count())
print('AQICN países/estaciones: consultar columna estacion para verificar alcance geográfico.')
if df_aqicn.count() == 100:
    print('AQICN: meta cuantitativa alcanzada con estaciones distintas (no histórico temporal).')


df_openmeteo: 100 filas, 8 columnas


df_aqicn: 100 filas, 8 columnas


df_mongo_bpm: 100 filas, 7 columnas


df_mysql_dht: 100 filas, 4 columnas


df_unificado: 100 filas, 28 columnas


Transformaciones y tres fuentes con 100 registros: verificadas.


OpenMeteo UV con datos: 100 de 100
AQICN países/estaciones: consultar columna estacion para verificar alcance geográfico.


AQICN: meta cuantitativa alcanzada con estaciones distintas (no histórico temporal).


## Declaración de uso de inteligencia artificial
De acuerdo con los lineamientos de la UNAB para el uso de IA generativa, declaramos que utilizamos un asistente de IA (Claude, de Anthropic) para revisar el código, detectar problemas de calidad de datos (como el índice UV vacío) y mejorar la documentación de este notebook. El grupo revisó, ejecutó y validó todas las conexiones, transformaciones y resultados, y es responsable de su contenido.